In [55]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
import re
import random
import time
import math
from tqdm import tqdm
import traceback

In [ ]:
MAX_REVIEWS_PER_PRODUCT = 150

# First review page of walmart product
PRODUCT_URLS = [
    "https://www.walmart.com/reviews/product/10451002?entryPoint=viewAllReviewsBottom",
    "https://www.walmart.com/reviews/product/5318328585?entryPoint=viewAllReviewsBottom",
    "https://www.walmart.com/reviews/product/5398899316?entryPoint=viewAllReviewsBottom"
]

In [57]:
from dataclasses import dataclass, field
from typing import Optional
import datetime

@dataclass
class Review:
    rating: Optional[float] = None
    reviewer_id: Optional[str] = None
    date: Optional[datetime.date] = None
    review_title: Optional[str] = None
    review_body: Optional[str] = None

@dataclass
class RatingDistribution:
    one_star: int = 0
    two_star: int = 0
    three_star: int = 0
    four_star: int = 0
    five_star: int = 0

@dataclass
class Product:
    name: str
    avg_rating: float
    reviews: list = field(default_factory=list)
    num_of_reviews: int = 0
    rating_distribution: Optional[RatingDistribution] = None


In [73]:
from collections import Counter

def scrape_walmart_reviews(product_url:str,driver:uc.Chrome) -> Product:
    try:
        
        # Load first page to find total number of pages
        driver.get(product_url)
        time.sleep(random.uniform(2, 4))
        # get the product details
        product_name = driver.find_element(By.XPATH,'//*[@id="maincontent"]/main/section/div[1]/div/span/span[2]/div').text.strip()
        product_avg_rating_str = driver.find_element(By.XPATH,'//*[@id="maincontent"]/main/section/div[2]/div[1]/section/div/span[2]')
        match = re.match(r'([\d.]+)', product_avg_rating_str.text.strip())
        product_avg_rating = float(match.group(1)) if match else 0.0

        product = Product(name=product_name, avg_rating=product_avg_rating)

        print(f"Scraping product: `{product.name}` with average rating of {product.avg_rating}")

        # Find the pagination element and get the last page number
        pagination_ul = driver.find_elements(By.CSS_SELECTOR, "ul.list.flex.items-center.justify-center.pa0")
        max_pages = 1
        
        if pagination_ul:
            li_items = pagination_ul[0].find_elements(By.TAG_NAME, "li")
            if len(li_items) > 1:
                # The second-to-last item is the last page number
                last_page_el = li_items[-2]
                last_page_text = last_page_el.text.strip()
                if last_page_text.isdigit():
                    max_pages = int(last_page_text)
                    print(f"Found {max_pages} pages total\n")
                else:
                    print(f"Could not parse page number: {last_page_text}")
                    raise Exception("Could not find max pages")

        # Cap pages based on MAX_REVIEWS_PER_PRODUCT (10 reviews per page)
        effective_max_pages = min(max_pages, math.ceil(MAX_REVIEWS_PER_PRODUCT / 10))

        for page_num in tqdm(range(1, effective_max_pages + 1), desc="Scraping pages", unit="page", ncols=100):

            # break if max reviews for this product is reached
            if len(product.reviews) >= MAX_REVIEWS_PER_PRODUCT:
                break
            
            # Parse the URL
            parsed_url = urlparse(product_url)
            query_params = parse_qs(parsed_url.query)
            query_params['page'] = [str(page_num)]
            new_query = urlencode(query_params, doseq=True)
            url = urlunparse(parsed_url._replace(query=new_query))

            if page_num > 1:
                driver.get(url)

            # Random wait to simulate human reading
            time.sleep(random.uniform(1, 2.5))

            # Simulate scrolling
            driver.execute_script(f"window.scrollTo(0, {random.randint(100, 500)});")
            time.sleep(random.uniform(0.5, 1))

            # Scrape all review cards (first card uses mt3-l, rest use mt4-l)
            review_cards = driver.find_elements(By.CSS_SELECTOR, ".overflow-visible.b--none.ma0.dark-gray")

            for card in tqdm(review_cards, desc=f"Page {page_num} cards", unit="card", leave=False, ncols=100):
                # Stars (first number from "x out of 5 stars")
                stars = None
                stars_els = card.find_elements(By.CSS_SELECTOR, ".ld_Ej")
                if stars_els:
                    match = re.match(r'([\d.]+)', stars_els[0].text.strip())
                    if match:
                        stars = float(match.group(1))

                # Reviewer ID
                reviewer_id = None
                reviewer_els = card.find_elements(By.CSS_SELECTOR, ".f7.b.mv0")
                if reviewer_els:
                    reviewer_id = reviewer_els[0].text.strip() or None

                # Title (can be null)
                title = None
                title_els = card.find_elements(By.CSS_SELECTOR, ".ld_EG.ld_EL.ld_EI.f5.b")
                if title_els:
                    title = title_els[0].text.strip() or None

                # Date
                date = None
                date_els = card.find_elements(By.CSS_SELECTOR, ".f7.gray.flex.flex-auto.flex-none-l.tr.tl-l.justify-end.justify-start-l")
                if date_els:
                    date = date_els[0].text.strip() or None

                # Review text
                review_text = None
                review_els = card.find_elements(By.CSS_SELECTOR, ".tl-m.db-m")
                if review_els:
                    review_text = review_els[0].text.strip() or None

                product.reviews.append(Review(
                    rating=stars,
                    reviewer_id=reviewer_id,
                    date=datetime.datetime.strptime(date, "%b %d, %Y").date() if date else None,
                    review_title=title,
                    review_body=review_text
                ))

                time.sleep(random.uniform(0.05, 0.15))

            # Random delay before going to next page to avoid bot detection
            if page_num < effective_max_pages:
                time.sleep(random.uniform(1, 2))

        # Compute rating distribution from scraped reviews
        counts = Counter(int(r.rating) for r in product.reviews if r.rating is not None)
        product.rating_distribution = RatingDistribution(
            one_star=counts.get(1, 0),
            two_star=counts.get(2, 0),
            three_star=counts.get(3, 0),
            four_star=counts.get(4, 0),
            five_star=counts.get(5, 0)
        )
        product.num_of_reviews = len(product.reviews)

        print(f"\nScraping complete! Collected {product.num_of_reviews} reviews.")
        return product
    except Exception as e:
        print(f"Exception while scraping {product_url}: {e}")
        traceback.print_exc()
        return None


In [74]:
# create driver and options for it
options = uc.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--lang=en-US')
options.add_argument('--start-maximized')
# Pin version_main to match your installed Chrome version
driver = uc.Chrome(options=options, headless=False, version_main=145)

In [75]:
# Main scraping loop
products :list[Product]= []
for product_url in PRODUCT_URLS:
    product = scrape_walmart_reviews(product_url,driver)
    if product:
        print(f"Product: {product.name} | Avg Rating: {product.avg_rating} | Reviews: {product.num_of_reviews} | Rating Distribution: {product.rating_distribution}")
    products.append(product)
driver.quit()

Scraping product: `Great Value Vegetable Oil, Heart Healthy and Versatile, 48 fl oz Bottle` with average rating of 4.7
Found 222 pages total



Scraping pages: 100%|█████████████████████████████████████████████| 15/15 [02:01<00:00,  8.10s/page]



Scraping complete! Collected 150 reviews.
Product: Great Value Vegetable Oil, Heart Healthy and Versatile, 48 fl oz Bottle | Avg Rating: 4.7 | Reviews: 150 | Rating Distribution: RatingDistribution(one_star=19, two_star=5, three_star=6, four_star=22, five_star=98)
Scraping product: `Zinus 6" Basic Tight Top Bonnell Innerspring Mattress, Twin` with average rating of 4.1
Found 135 pages total



Scraping pages: 100%|█████████████████████████████████████████████| 15/15 [01:43<00:00,  6.91s/page]



Scraping complete! Collected 140 reviews.
Product: Zinus 6" Basic Tight Top Bonnell Innerspring Mattress, Twin | Avg Rating: 4.1 | Reviews: 140 | Rating Distribution: RatingDistribution(one_star=59, two_star=10, three_star=17, four_star=8, five_star=46)
Scraping product: `ARTSEA Projector 4K 1080P Support, Mini Projector Smart for Movie Projection Compatible with Phone- White` with average rating of 3.9
Found 88 pages total



Scraping pages: 100%|█████████████████████████████████████████████| 15/15 [02:01<00:00,  8.12s/page]



Scraping complete! Collected 150 reviews.
Product: ARTSEA Projector 4K 1080P Support, Mini Projector Smart for Movie Projection Compatible with Phone- White | Avg Rating: 3.9 | Reviews: 150 | Rating Distribution: RatingDistribution(one_star=37, two_star=13, three_star=17, four_star=25, five_star=58)


In [76]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("scraped_data")
OUTPUT_DIR.mkdir(exist_ok=True)

def sanitize_folder_name(name: str) -> str:
    name = re.sub(r'[^\w\s-]', '', name)
    name = re.sub(r'[\s]+', '_', name.strip())
    return name[:80]  # cap length

# Save products.csv (metadata)
products_rows = []
for p in products:
    if p is None:
        continue
    rd = p.rating_distribution
    products_rows.append({
        "name": p.name,
        "avg_rating": p.avg_rating,
        "num_of_reviews": p.num_of_reviews,
        "one_star":   rd.one_star   if rd else 0,
        "two_star":   rd.two_star   if rd else 0,
        "three_star": rd.three_star if rd else 0,
        "four_star":  rd.four_star  if rd else 0,
        "five_star":  rd.five_star  if rd else 0,
    })

products_df = pd.DataFrame(products_rows)
products_df.to_csv(OUTPUT_DIR / "products.csv", index=False)
print(f"Saved products metadata → {OUTPUT_DIR / 'products.csv'}")

# Save per-product reviews.csv
for p in products:
    if p is None:
        continue
    folder_name = sanitize_folder_name(p.name)
    product_dir = OUTPUT_DIR / folder_name
    product_dir.mkdir(exist_ok=True)

    reviews_rows = [
        {
            "rating":       r.rating,
            "reviewer_id":  r.reviewer_id,
            "date":         r.date,
            "review_title": r.review_title,
            "review_body":  r.review_body,
        }
        for r in p.reviews
    ]
    reviews_df = pd.DataFrame(reviews_rows)
    reviews_path = product_dir / "reviews.csv"
    reviews_df.to_csv(reviews_path, index=False)
    print(f"Saved {len(reviews_df)} reviews → {reviews_path}")


Saved products metadata → scraped_data/products.csv
Saved 150 reviews → scraped_data/Great_Value_Vegetable_Oil_Heart_Healthy_and_Versatile_48_fl_oz_Bottle/reviews.csv
Saved 140 reviews → scraped_data/Zinus_6_Basic_Tight_Top_Bonnell_Innerspring_Mattress_Twin/reviews.csv
Saved 150 reviews → scraped_data/ARTSEA_Projector_4K_1080P_Support_Mini_Projector_Smart_for_Movie_Projection_Comp/reviews.csv
